In [4]:
import requests
from lakehouse.daft import bronze, silver
from lakehouse.daft.utils import daftutils
import json
import daft
from deltalake.table import TableMerger, DeltaTable


In [5]:
CATALOG = "daft_catalog"

# 1. Set Up and Bronze Data

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
@daft.udf(return_dtype=daft.DataType.string())
def get_properties(urls: daft.Series) -> list:
    result = []
    for url in urls.to_pylist():
        json_request = requests.get(url).json()
        result.append(json.dumps(json_request["result"]["properties"]))
    return result

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return daft.from_pylist(results)

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.with_column("properties", get_properties(daft.col("url")))
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


bronze_instance = StarWarsBronze(**options)

In [9]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-15 22:47:02 | people | execute | Started
2025-03-15 22:47:02 | people | load | Started
2025-03-15 22:47:07 | people | load | Completed in 0.07 min
2025-03-15 22:47:07 | people | transform | Started
2025-03-15 22:47:07 | people | transform | Completed in 0.0 min
2025-03-15 22:47:07 | people | write | Started
c:\Users\nikol\miniconda3\envs\pyspark3-exec\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


                                                           d

2025-03-15 22:47:41 | people | write | Completed in 0.57 min
2025-03-15 22:47:41 | people | execute | Completed in 0.63 min
2025-03-15 22:47:41 | planets | execute | Started
2025-03-15 22:47:41 | planets | load | Started


2025-03-15 22:47:44 | planets | load | Completed in 0.05 min
2025-03-15 22:47:44 | planets | transform | Started
2025-03-15 22:47:44 | planets | transform | Completed in 0.0 min
2025-03-15 22:47:44 | planets | write | Started


                                                           d

2025-03-15 22:48:08 | planets | write | Completed in 0.38 min
2025-03-15 22:48:08 | planets | execute | Completed in 0.43 min


In [10]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:47:07.220900,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:47:07.220900,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:47:07.220900,R2-D2,3,https://www.swapi.tech/api/people/3,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R2-D2"", ""gender"": ""n/a"", ""skin_color"": ""white, blue"", ""hair_color"": ""n/a"", ""height"": ""96"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""33BBY"", ""url"": ""https://www.swapi.tech/api/people/3""}"
2025-03-15 22:47:07.220900,Darth Vader,4,https://www.swapi.tech/api/people/4,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Darth Vader"", ""gender"": ""male"", ""skin_color"": ""white"", ""hair_color"": ""none"", ""height"": ""202"", ""eye_color"": ""yellow"", ""mass"": ""136"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/4""}"
2025-03-15 22:47:07.220900,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-15 22:47:07.220900,Owen Lars,6,https://www.swapi.tech/api/people/6,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Owen Lars"", ""gender"": ""male"", ""skin_color"": ""light"", ""hair_color"": ""brown, grey"", ""height"": ""178"", ""eye_color"": ""blue"", ""mass"": ""120"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""52BBY"", ""url"": ""https://www.swapi.tech/api/people/6""}"
2025-03-15 22:47:07.220900,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Beru Whitesun lars"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""165"", ""eye_color"": ""blue"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""47BBY"", ""url"": ""https://www.swapi.tech/api/people/7""}"
2025-03-15 22:47:07.220900,R5-D4,8,https://www.swapi.tech/api/people/8,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R5-D4"", ""gender"": ""n/a"", ""skin_color"": ""white, red"", ""hair_color"": ""n/a"", ""height"": ""97"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/8""}"


No. Rows: 82


In [11]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/planets")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:47:44.286978,Tatooine,1,https://www.swapi.tech/api/planets/1,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""arid"", ""surface_water"": ""1"", ""name"": ""Tatooine"", ""diameter"": ""10465"", ""rotation_period"": ""23"", ""terrain"": ""desert"", ""gravity"": ""1 standard"", ""orbital_period"": ""304"", ""population"": ""200000"", ""url"": ""https://www.swapi.tech/api/planets/1""}"
2025-03-15 22:47:44.286978,Alderaan,2,https://www.swapi.tech/api/planets/2,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""40"", ""name"": ""Alderaan"", ""diameter"": ""12500"", ""rotation_period"": ""24"", ""terrain"": ""grasslands, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""364"", ""population"": ""2000000000"", ""url"": ""https://www.swapi.tech/api/planets/2""}"
2025-03-15 22:47:44.286978,Yavin IV,3,https://www.swapi.tech/api/planets/3,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate, tropical"", ""surface_water"": ""8"", ""name"": ""Yavin IV"", ""diameter"": ""10200"", ""rotation_period"": ""24"", ""terrain"": ""jungle, rainforests"", ""gravity"": ""1 standard"", ""orbital_period"": ""4818"", ""population"": ""1000"", ""url"": ""https://www.swapi.tech/api/planets/3""}"
2025-03-15 22:47:44.286978,Hoth,4,https://www.swapi.tech/api/planets/4,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""frozen"", ""surface_water"": ""100"", ""name"": ""Hoth"", ""diameter"": ""7200"", ""rotation_period"": ""23"", ""terrain"": ""tundra, ice caves, mountain ranges"", ""gravity"": ""1.1 standard"", ""orbital_period"": ""549"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/4""}"
2025-03-15 22:47:44.286978,Dagobah,5,https://www.swapi.tech/api/planets/5,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""murky"", ""surface_water"": ""8"", ""name"": ""Dagobah"", ""diameter"": ""8900"", ""rotation_period"": ""23"", ""terrain"": ""swamp, jungles"", ""gravity"": ""N/A"", ""orbital_period"": ""341"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/5""}"
2025-03-15 22:47:44.286978,Bespin,6,https://www.swapi.tech/api/planets/6,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""0"", ""name"": ""Bespin"", ""diameter"": ""118000"", ""rotation_period"": ""12"", ""terrain"": ""gas giant"", ""gravity"": ""1.5 (surface), 1 standard (Cloud City)"", ""orbital_period"": ""5110"", ""population"": ""6000000"", ""url"": ""https://www.swapi.tech/api/planets/6""}"
2025-03-15 22:47:44.286978,Endor,7,https://www.swapi.tech/api/planets/7,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""8"", ""name"": ""Endor"", ""diameter"": ""4900"", ""rotation_period"": ""18"", ""terrain"": ""forests, mountains, lakes"", ""gravity"": ""0.85 standard"", ""orbital_period"": ""402"", ""population"": ""30000000"", ""url"": ""https://www.swapi.tech/api/planets/7""}"
2025-03-15 22:47:44.286978,Naboo,8,https://www.swapi.tech/api/planets/8,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""12"", ""name"": ""Naboo"", ""diameter"": ""12120"", ""rotation_period"": ""26"", ""terrain"": ""grassy hills, swamps, forests, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""312"", ""population"": ""4500000000"", ""url"": ""https://www.swapi.tech/api/planets/8""}"


No. Rows: 60


In [12]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [13]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        df = df.with_column("uid", daft.col("uid").cast(daftutils.get_daft_dtype("int")))
        return df

    def source_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.source_schema}/{table}"
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"

silver_instance = StarWarsSilver(
    catalog=CATALOG, source_schema="bronze", target_schema="silver"
)

In [14]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute(
    "people", "planets"
)

2025-03-15 22:48:08 | people | execute | Started
2025-03-15 22:48:08 | people | load | Started
2025-03-15 22:48:08 | people | load | Completed in 0.0 min
2025-03-15 22:48:08 | people | transform | Started
2025-03-15 22:48:08 | people | transform | Completed in 0.0 min
2025-03-15 22:48:08 | people | write | Started
2025-03-15 22:48:08 | people | write | Completed in 0.0 min
2025-03-15 22:48:08 | people | execute | Completed in 0.0 min
2025-03-15 22:48:08 | planets | execute | Started
2025-03-15 22:48:08 | planets | load | Started
2025-03-15 22:48:08 | planets | load | Completed in 0.0 min
2025-03-15 22:48:08 | planets | transform | Started
2025-03-15 22:48:08 | planets | transform | Completed in 0.0 min
2025-03-15 22:48:08 | planets | write | Started
2025-03-15 22:48:08 | planets | write | Completed in 0.0 min
2025-03-15 22:48:08 | planets | execute | Completed in 0.0 min


In [15]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,propertiesUtf8
2025-03-15 22:48:08.322093,2025-03-15 22:47:07.220900,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:48:08.322093,2025-03-15 22:47:07.220900,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:48:08.322093,2025-03-15 22:47:07.220900,Obi-Wan Kenobi,10,https://www.swapi.tech/api/people/10,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Obi-Wan Kenobi"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""auburn, white"", ""height"": ""182"", ""eye_color"": ""blue-gray"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/20"", ""birth_year"": ""57BBY"", ""url"": ""https://www.swapi.tech/api/people/10""}"
2025-03-15 22:48:08.322093,2025-03-15 22:47:07.220900,Anakin Skywalker,11,https://www.swapi.tech/api/people/11,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Anakin Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""188"", ""eye_color"": ""blue"", ""mass"": ""84"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/11""}"
2025-03-15 22:48:08.322093,2025-03-15 22:47:07.220900,Wilhuff Tarkin,12,https://www.swapi.tech/api/people/12,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Wilhuff Tarkin"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""auburn, grey"", ""height"": ""180"", ""eye_color"": ""blue"", ""mass"": ""unknown"", ""homeworld"": ""https://www.swapi.tech/api/planets/21"", ""birth_year"": ""64BBY"", ""url"": ""https://www.swapi.tech/api/people/12""}"
2025-03-15 22:48:08.322093,2025-03-15 22:47:07.220900,Chewbacca,13,https://www.swapi.tech/api/people/13,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Chewbacca"", ""gender"": ""male"", ""skin_color"": ""unknown"", ""hair_color"": ""brown"", ""height"": ""228"", ""eye_color"": ""blue"", ""mass"": ""112"", ""homeworld"": ""https://www.swapi.tech/api/planets/14"", ""birth_year"": ""200BBY"", ""url"": ""https://www.swapi.tech/api/people/13""}"
2025-03-15 22:48:08.322093,2025-03-15 22:47:07.220900,Han Solo,14,https://www.swapi.tech/api/people/14,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Han Solo"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""brown"", ""height"": ""180"", ""eye_color"": ""brown"", ""mass"": ""80"", ""homeworld"": ""https://www.swapi.tech/api/planets/22"", ""birth_year"": ""29BBY"", ""url"": ""https://www.swapi.tech/api/people/14""}"
2025-03-15 22:48:08.322093,2025-03-15 22:47:07.220900,Greedo,15,https://www.swapi.tech/api/people/15,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Greedo"", ""gender"": ""male"", ""skin_color"": ""green"", ""hair_color"": ""n/a"", ""height"": ""173"", ""eye_color"": ""black"", ""mass"": ""74"", ""homeworld"": ""https

No. Rows: 17


In [16]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/planets")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,propertiesUtf8
2025-03-15 22:48:08.380619,2025-03-15 22:47:44.286978,Tatooine,1,https://www.swapi.tech/api/planets/1,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""arid"", ""surface_water"": ""1"", ""name"": ""Tatooine"", ""diameter"": ""10465"", ""rotation_period"": ""23"", ""terrain"": ""desert"", ""gravity"": ""1 standard"", ""orbital_period"": ""304"", ""population"": ""200000"", ""url"": ""https://www.swapi.tech/api/planets/1""}"
2025-03-15 22:48:08.380619,2025-03-15 22:47:44.286978,Alderaan,2,https://www.swapi.tech/api/planets/2,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""40"", ""name"": ""Alderaan"", ""diameter"": ""12500"", ""rotation_period"": ""24"", ""terrain"": ""grasslands, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""364"", ""population"": ""2000000000"", ""url"": ""https://www.swapi.tech/api/planets/2""}"
2025-03-15 22:48:08.380619,2025-03-15 22:47:44.286978,Kamino,10,https://www.swapi.tech/api/planets/10,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""100"", ""name"": ""Kamino"", ""diameter"": ""19720"", ""rotation_period"": ""27"", ""terrain"": ""ocean"", ""gravity"": ""1 standard"", ""orbital_period"": ""463"", ""population"": ""1000000000"", ""url"": ""https://www.swapi.tech/api/planets/10""}"
2025-03-15 22:48:08.380619,2025-03-15 22:47:44.286978,Geonosis,11,https://www.swapi.tech/api/planets/11,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate, arid"", ""surface_water"": ""5"", ""name"": ""Geonosis"", ""diameter"": ""11370"", ""rotation_period"": ""30"", ""terrain"": ""rock, desert, mountain, barren"", ""gravity"": ""0.9 standard"", ""orbital_period"": ""256"", ""population"": ""100000000000"", ""url"": ""https://www.swapi.tech/api/planets/11""}"
2025-03-15 22:48:08.380619,2025-03-15 22:47:44.286978,Utapau,12,https://www.swapi.tech/api/planets/12,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate, arid, windy"", ""surface_water"": ""0.9"", ""name"": ""Utapau"", ""diameter"": ""12900"", ""rotation_period"": ""27"", ""terrain"": ""scrublands, savanna, canyons, sinkholes"", ""gravity"": ""1 standard"", ""orbital_period"": ""351"", ""population"": ""95000000"", ""url"": ""https://www.swapi.tech/api/planets/12""}"
2025-03-15 22:48:08.380619,2025-03-15 22:47:44.286978,Mustafar,13,https://www.swapi.tech/api/planets/13,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""hot"", ""surface_water"": ""0"", ""name"": ""Mustafar"", ""diameter"": ""4200"", ""rotation_period"": ""36"", ""terrain"": ""volcanoes, lava rivers, mountains, caves"", ""gravity"": ""1 standard"", ""orbital_period"": ""412"", ""population"": ""20000"", ""url"": ""https://www.swapi.tech/api/planets/13""}"
2025-03-15 22:48:08.380619,2025-03-15 22:47:44.286978,Kashyyyk,14,https://www.swapi.tech/api/planets/14,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""tropical"", ""surface_water"": ""60"", ""name"": ""Kashyyyk"", ""diameter"": ""12765"", ""rotation_period"": ""26"", ""terrain"": ""jungle, forests, lakes, rivers"", ""gravity"": ""1 standard"", ""orbital_period"": ""381"", ""population"": ""45000000"", ""url"": ""https://www.swapi.tech/api/planets/14""}"
2025-03-15 22:48:08.380619,2025-03-15 22:47:44.286978,Polis Massa,15,https://www.swapi.tech/api/planets/15,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""artificial temperate "", ""surface_water"": ""0"", ""name"": ""Polis Massa"", ""diameter"": ""0"", ""rotation

No. Rows: 18


In [17]:
# without load filter
silver_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-15 22:48:08 | people | execute | Started
2025-03-15 22:48:08 | people | load | Started
2025-03-15 22:48:08 | people | load | Completed in 0.0 min
2025-03-15 22:48:08 | people | transform | Started
2025-03-15 22:48:08 | people | transform | Completed in 0.0 min
2025-03-15 22:48:08 | people | write | Started
2025-03-15 22:48:08 | people | write | Completed in 0.0 min
2025-03-15 22:48:08 | people | execute | Completed in 0.0 min
2025-03-15 22:48:08 | planets | execute | Started
2025-03-15 22:48:08 | planets | load | Started
2025-03-15 22:48:08 | planets | load | Completed in 0.0 min
2025-03-15 22:48:08 | planets | transform | Started
2025-03-15 22:48:08 | planets | transform | Completed in 0.0 min
2025-03-15 22:48:08 | planets | write | Started
2025-03-15 22:48:09 | planets | write | Completed in 0.02 min
2025-03-15 22:48:09 | planets | execute | Completed in 0.02 min


In [18]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,propertiesUtf8
2025-03-15 22:48:08.490892,2025-03-15 22:47:07.220900,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:48:08.490892,2025-03-15 22:47:07.220900,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:48:08.490892,2025-03-15 22:47:07.220900,R2-D2,3,https://www.swapi.tech/api/people/3,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R2-D2"", ""gender"": ""n/a"", ""skin_color"": ""white, blue"", ""hair_color"": ""n/a"", ""height"": ""96"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""33BBY"", ""url"": ""https://www.swapi.tech/api/people/3""}"
2025-03-15 22:48:08.490892,2025-03-15 22:47:07.220900,Darth Vader,4,https://www.swapi.tech/api/people/4,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Darth Vader"", ""gender"": ""male"", ""skin_color"": ""white"", ""hair_color"": ""none"", ""height"": ""202"", ""eye_color"": ""yellow"", ""mass"": ""136"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/4""}"
2025-03-15 22:48:08.490892,2025-03-15 22:47:07.220900,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-15 22:48:08.490892,2025-03-15 22:47:07.220900,Owen Lars,6,https://www.swapi.tech/api/people/6,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Owen Lars"", ""gender"": ""male"", ""skin_color"": ""light"", ""hair_color"": ""brown, grey"", ""height"": ""178"", ""eye_color"": ""blue"", ""mass"": ""120"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""52BBY"", ""url"": ""https://www.swapi.tech/api/people/6""}"
2025-03-15 22:48:08.490892,2025-03-15 22:47:07.220900,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Beru Whitesun lars"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""165"", ""eye_color"": ""blue"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""47BBY"", ""url"": ""https://www.swapi.tech/api/people/7""}"
2025-03-15 22:48:08.490892,2025-03-15 22:47:07.220900,R5-D4,8,https://www.swapi.tech/api/people/8,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R5-D4"", ""gender"": ""n/a"", ""skin_color"": ""white, red"", ""hair_color"": ""n/a"", ""height"": ""97"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year

No. Rows: 82


In [19]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/planets")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,propertiesUtf8
2025-03-15 22:48:08.562241,2025-03-15 22:47:44.286978,Tatooine,1,https://www.swapi.tech/api/planets/1,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""arid"", ""surface_water"": ""1"", ""name"": ""Tatooine"", ""diameter"": ""10465"", ""rotation_period"": ""23"", ""terrain"": ""desert"", ""gravity"": ""1 standard"", ""orbital_period"": ""304"", ""population"": ""200000"", ""url"": ""https://www.swapi.tech/api/planets/1""}"
2025-03-15 22:48:08.562241,2025-03-15 22:47:44.286978,Alderaan,2,https://www.swapi.tech/api/planets/2,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""40"", ""name"": ""Alderaan"", ""diameter"": ""12500"", ""rotation_period"": ""24"", ""terrain"": ""grasslands, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""364"", ""population"": ""2000000000"", ""url"": ""https://www.swapi.tech/api/planets/2""}"
2025-03-15 22:48:08.562241,2025-03-15 22:47:44.286978,Yavin IV,3,https://www.swapi.tech/api/planets/3,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate, tropical"", ""surface_water"": ""8"", ""name"": ""Yavin IV"", ""diameter"": ""10200"", ""rotation_period"": ""24"", ""terrain"": ""jungle, rainforests"", ""gravity"": ""1 standard"", ""orbital_period"": ""4818"", ""population"": ""1000"", ""url"": ""https://www.swapi.tech/api/planets/3""}"
2025-03-15 22:48:08.562241,2025-03-15 22:47:44.286978,Hoth,4,https://www.swapi.tech/api/planets/4,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""frozen"", ""surface_water"": ""100"", ""name"": ""Hoth"", ""diameter"": ""7200"", ""rotation_period"": ""23"", ""terrain"": ""tundra, ice caves, mountain ranges"", ""gravity"": ""1.1 standard"", ""orbital_period"": ""549"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/4""}"
2025-03-15 22:48:08.562241,2025-03-15 22:47:44.286978,Dagobah,5,https://www.swapi.tech/api/planets/5,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""murky"", ""surface_water"": ""8"", ""name"": ""Dagobah"", ""diameter"": ""8900"", ""rotation_period"": ""23"", ""terrain"": ""swamp, jungles"", ""gravity"": ""N/A"", ""orbital_period"": ""341"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/5""}"
2025-03-15 22:48:08.562241,2025-03-15 22:47:44.286978,Bespin,6,https://www.swapi.tech/api/planets/6,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""0"", ""name"": ""Bespin"", ""diameter"": ""118000"", ""rotation_period"": ""12"", ""terrain"": ""gas giant"", ""gravity"": ""1.5 (surface), 1 standard (Cloud City)"", ""orbital_period"": ""5110"", ""population"": ""6000000"", ""url"": ""https://www.swapi.tech/api/planets/6""}"
2025-03-15 22:48:08.562241,2025-03-15 22:47:44.286978,Endor,7,https://www.swapi.tech/api/planets/7,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""8"", ""name"": ""Endor"", ""diameter"": ""4900"", ""rotation_period"": ""18"", ""terrain"": ""forests, mountains, lakes"", ""gravity"": ""0.85 standard"", ""orbital_period"": ""402"", ""population"": ""30000000"", ""url"": ""https://www.swapi.tech/api/planets/7""}"
2025-03-15 22:48:08.562241,2025-03-15 22:47:44.286978,Naboo,8,https://www.swapi.tech/api/planets/8,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""12"", ""name"": ""Naboo"", ""diameter"": ""12120"", ""rotation_period"": ""26"", ""terrain"": ""grassy hills, swamps, forests, mountains"", ""gravity"

No. Rows: 60


# 3 Replace Where

In [20]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        df = df.with_column("uid", daft.col("uid").cast(daftutils.get_daft_dtype("int")))
        return df

    def source_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.source_schema}/{table}"
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"
    
    def get_replace_condition(self, df: daft.DataFrame, table: str) -> str:
        return "uid > 0"

silver_instance = StarWarsSilver(
    catalog=CATALOG, source_schema="bronze", target_schema="silver"
)

silver_instance.load(filter="custom").transform().write(mode="replace").execute(
    "people", "planets"
)

2025-03-15 22:48:09 | people | execute | Started
2025-03-15 22:48:09 | people | load | Started
2025-03-15 22:48:09 | people | load | Completed in 0.0 min
2025-03-15 22:48:09 | people | transform | Started
2025-03-15 22:48:09 | people | transform | Completed in 0.0 min
2025-03-15 22:48:09 | people | write | Started
2025-03-15 22:48:10 | people | write | Completed in 0.0 min
2025-03-15 22:48:10 | people | execute | Completed in 0.0 min
2025-03-15 22:48:10 | planets | execute | Started
2025-03-15 22:48:10 | planets | load | Started
2025-03-15 22:48:10 | planets | load | Completed in 0.0 min
2025-03-15 22:48:10 | planets | transform | Started
2025-03-15 22:48:10 | planets | transform | Completed in 0.0 min
2025-03-15 22:48:10 | planets | write | Started
2025-03-15 22:48:10 | planets | write | Completed in 0.0 min
2025-03-15 22:48:10 | planets | execute | Completed in 0.0 min


In [21]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,propertiesUtf8
2025-03-15 22:48:09.876354,2025-03-15 22:47:07.220900,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:48:09.876354,2025-03-15 22:47:07.220900,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:48:09.876354,2025-03-15 22:47:07.220900,Obi-Wan Kenobi,10,https://www.swapi.tech/api/people/10,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Obi-Wan Kenobi"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""auburn, white"", ""height"": ""182"", ""eye_color"": ""blue-gray"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/20"", ""birth_year"": ""57BBY"", ""url"": ""https://www.swapi.tech/api/people/10""}"
2025-03-15 22:48:09.876354,2025-03-15 22:47:07.220900,Anakin Skywalker,11,https://www.swapi.tech/api/people/11,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Anakin Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""188"", ""eye_color"": ""blue"", ""mass"": ""84"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/11""}"
2025-03-15 22:48:09.876354,2025-03-15 22:47:07.220900,Wilhuff Tarkin,12,https://www.swapi.tech/api/people/12,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Wilhuff Tarkin"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""auburn, grey"", ""height"": ""180"", ""eye_color"": ""blue"", ""mass"": ""unknown"", ""homeworld"": ""https://www.swapi.tech/api/planets/21"", ""birth_year"": ""64BBY"", ""url"": ""https://www.swapi.tech/api/people/12""}"
2025-03-15 22:48:09.876354,2025-03-15 22:47:07.220900,Chewbacca,13,https://www.swapi.tech/api/people/13,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Chewbacca"", ""gender"": ""male"", ""skin_color"": ""unknown"", ""hair_color"": ""brown"", ""height"": ""228"", ""eye_color"": ""blue"", ""mass"": ""112"", ""homeworld"": ""https://www.swapi.tech/api/planets/14"", ""birth_year"": ""200BBY"", ""url"": ""https://www.swapi.tech/api/people/13""}"
2025-03-15 22:48:09.876354,2025-03-15 22:47:07.220900,Han Solo,14,https://www.swapi.tech/api/people/14,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Han Solo"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""brown"", ""height"": ""180"", ""eye_color"": ""brown"", ""mass"": ""80"", ""homeworld"": ""https://www.swapi.tech/api/planets/22"", ""birth_year"": ""29BBY"", ""url"": ""https://www.swapi.tech/api/people/14""}"
2025-03-15 22:48:09.876354,2025-03-15 22:47:07.220900,Greedo,15,https://www.swapi.tech/api/people/15,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Greedo"", ""gender"": ""male"", ""skin_color"": ""green"", ""hair_color"": ""n/a"", ""height"": ""173"", ""eye_color"": ""black"", ""mass"": ""74"", ""homeworld"": ""https

No. Rows: 17


# 4 Append

In [22]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        df = df.with_column("uid", daft.col("uid").cast(daftutils.get_daft_dtype("int")))
        return df

    def source_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.source_schema}/{table}"
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"

silver_instance = StarWarsSilver(
    catalog=CATALOG, source_schema="bronze", target_schema="silver"
)
silver_instance.load().transform().write().execute(
    "people", "planets"
)  # default mode is append

2025-03-15 22:48:10 | people | execute | Started
2025-03-15 22:48:10 | people | load | Started
2025-03-15 22:48:10 | people | load | Completed in 0.0 min
2025-03-15 22:48:10 | people | transform | Started
2025-03-15 22:48:10 | people | transform | Completed in 0.0 min
2025-03-15 22:48:10 | people | write | Started
2025-03-15 22:48:10 | people | write | Completed in 0.0 min
2025-03-15 22:48:10 | people | execute | Completed in 0.0 min
2025-03-15 22:48:10 | planets | execute | Started
2025-03-15 22:48:10 | planets | load | Started
2025-03-15 22:48:10 | planets | load | Completed in 0.0 min
2025-03-15 22:48:10 | planets | transform | Started
2025-03-15 22:48:10 | planets | transform | Completed in 0.0 min
2025-03-15 22:48:10 | planets | write | Started
2025-03-15 22:48:10 | planets | write | Completed in 0.0 min
2025-03-15 22:48:10 | planets | execute | Completed in 0.0 min


In [23]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,propertiesUtf8
2025-03-15 22:48:10.118569,2025-03-15 22:47:07.220900,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:48:10.118569,2025-03-15 22:47:07.220900,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:48:10.118569,2025-03-15 22:47:07.220900,R2-D2,3,https://www.swapi.tech/api/people/3,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R2-D2"", ""gender"": ""n/a"", ""skin_color"": ""white, blue"", ""hair_color"": ""n/a"", ""height"": ""96"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""33BBY"", ""url"": ""https://www.swapi.tech/api/people/3""}"
2025-03-15 22:48:10.118569,2025-03-15 22:47:07.220900,Darth Vader,4,https://www.swapi.tech/api/people/4,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Darth Vader"", ""gender"": ""male"", ""skin_color"": ""white"", ""hair_color"": ""none"", ""height"": ""202"", ""eye_color"": ""yellow"", ""mass"": ""136"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/4""}"
2025-03-15 22:48:10.118569,2025-03-15 22:47:07.220900,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-15 22:48:10.118569,2025-03-15 22:47:07.220900,Owen Lars,6,https://www.swapi.tech/api/people/6,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Owen Lars"", ""gender"": ""male"", ""skin_color"": ""light"", ""hair_color"": ""brown, grey"", ""height"": ""178"", ""eye_color"": ""blue"", ""mass"": ""120"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""52BBY"", ""url"": ""https://www.swapi.tech/api/people/6""}"
2025-03-15 22:48:10.118569,2025-03-15 22:47:07.220900,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Beru Whitesun lars"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""165"", ""eye_color"": ""blue"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""47BBY"", ""url"": ""https://www.swapi.tech/api/people/7""}"
2025-03-15 22:48:10.118569,2025-03-15 22:47:07.220900,R5-D4,8,https://www.swapi.tech/api/people/8,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R5-D4"", ""gender"": ""n/a"", ""skin_color"": ""white, red"", ""hair_color"": ""n/a"", ""height"": ""97"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year

No. Rows: 99


# 5 Merge

In [24]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        df = df.with_column("uid", daft.col("uid").cast(daftutils.get_daft_dtype("int")))
        return df

    def source_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.source_schema}/{table}"
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"

    def get_delta_merge_builder(
        self, df: daft.DataFrame, delta_table: DeltaTable
    ) -> TableMerger:
        merge_condition = "target.url = source.url"
        builder = delta_table.merge(df, predicate=merge_condition, source_alias="source", target_alias="target")
        builder = builder.when_matched_update_all()
        builder = builder.when_not_matched_insert_all()
        return builder

silver_instance = StarWarsSilver(
    catalog=CATALOG, source_schema="bronze", target_schema="silver"
)

silver_instance.load().transform().write(mode="merge").execute("people", "planets")

2025-03-15 22:48:10 | people | execute | Started
2025-03-15 22:48:10 | people | load | Started
2025-03-15 22:48:10 | people | load | Completed in 0.0 min
2025-03-15 22:48:10 | people | transform | Started
2025-03-15 22:48:10 | people | transform | Completed in 0.0 min
2025-03-15 22:48:10 | people | write | Started
2025-03-15 22:48:11 | people | write | Completed in 0.02 min
2025-03-15 22:48:11 | people | execute | Completed in 0.02 min
2025-03-15 22:48:11 | planets | execute | Started
2025-03-15 22:48:11 | planets | load | Started
2025-03-15 22:48:11 | planets | load | Completed in 0.0 min
2025-03-15 22:48:11 | planets | transform | Started
2025-03-15 22:48:11 | planets | transform | Completed in 0.0 min
2025-03-15 22:48:11 | planets | write | Started
2025-03-15 22:48:11 | planets | write | Completed in 0.0 min
2025-03-15 22:48:11 | planets | execute | Completed in 0.0 min


In [25]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,propertiesUtf8
2025-03-15 22:48:10.279821,2025-03-15 22:47:07.220900,Obi-Wan Kenobi,10,https://www.swapi.tech/api/people/10,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Obi-Wan Kenobi"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""auburn, white"", ""height"": ""182"", ""eye_color"": ""blue-gray"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/20"", ""birth_year"": ""57BBY"", ""url"": ""https://www.swapi.tech/api/people/10""}"
2025-03-15 22:48:10.279821,2025-03-15 22:47:07.220900,Anakin Skywalker,11,https://www.swapi.tech/api/people/11,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Anakin Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""188"", ""eye_color"": ""blue"", ""mass"": ""84"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/11""}"
2025-03-15 22:48:10.279821,2025-03-15 22:47:07.220900,Boba Fett,22,https://www.swapi.tech/api/people/22,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Boba Fett"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""black"", ""height"": ""183"", ""eye_color"": ""brown"", ""mass"": ""78.2"", ""homeworld"": ""https://www.swapi.tech/api/planets/10"", ""birth_year"": ""31.5BBY"", ""url"": ""https://www.swapi.tech/api/people/22""}"
2025-03-15 22:48:10.279821,2025-03-15 22:47:07.220900,Lando Calrissian,25,https://www.swapi.tech/api/people/25,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Lando Calrissian"", ""gender"": ""male"", ""skin_color"": ""dark"", ""hair_color"": ""black"", ""height"": ""177"", ""eye_color"": ""brown"", ""mass"": ""79"", ""homeworld"": ""https://www.swapi.tech/api/planets/30"", ""birth_year"": ""31BBY"", ""url"": ""https://www.swapi.tech/api/people/25""}"
2025-03-15 22:48:10.279821,2025-03-15 22:47:07.220900,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:48:10.279821,2025-03-15 22:47:07.220900,Jek Tono Porkins,19,https://www.swapi.tech/api/people/19,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Jek Tono Porkins"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""brown"", ""height"": ""180"", ""eye_color"": ""blue"", ""mass"": ""110"", ""homeworld"": ""https://www.swapi.tech/api/planets/26"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/19""}"
2025-03-15 22:48:10.279821,2025-03-15 22:47:07.220900,Wilhuff Tarkin,12,https://www.swapi.tech/api/people/12,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Wilhuff Tarkin"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""auburn, grey"", ""height"": ""180"", ""eye_color"": ""blue"", ""mass"": ""unknown"", ""homeworld"": ""https://www.swapi.tech/api/planets/21"", ""birth_year"": ""64BBY"", ""url"": ""https://www.swapi.tech/api/people/12""}"
2025-03-15 22:48:10.279821,2025-03-15 22:47:07.220900,Greedo,15,https://www.swapi.tech/api/people/15,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Greedo"", ""gender"": ""male"", ""skin_color"": ""green"", ""hair_color"": ""n/a"", ""height"": ""173"", ""eye_color"": ""blac

No. Rows: 99


In [26]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/planets")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,propertiesUtf8
2025-03-15 22:48:11.471530,2025-03-15 22:47:44.286978,Ryloth,37,https://www.swapi.tech/api/planets/37,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate, arid, subartic"", ""surface_water"": ""5"", ""name"": ""Ryloth"", ""diameter"": ""10600"", ""rotation_period"": ""30"", ""terrain"": ""mountains, valleys, deserts, tundra"", ""gravity"": ""1"", ""orbital_period"": ""305"", ""population"": ""1500000000"", ""url"": ""https://www.swapi.tech/api/planets/37""}"
2025-03-15 22:48:11.471530,2025-03-15 22:47:44.286978,Trandosha,29,https://www.swapi.tech/api/planets/29,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""arid"", ""surface_water"": ""unknown"", ""name"": ""Trandosha"", ""diameter"": ""0"", ""rotation_period"": ""25"", ""terrain"": ""mountains, seas, grasslands, deserts"", ""gravity"": ""0.62 standard"", ""orbital_period"": ""371"", ""population"": ""42000000"", ""url"": ""https://www.swapi.tech/api/planets/29""}"
2025-03-15 22:48:11.471530,2025-03-15 22:47:44.286978,Tatooine,1,https://www.swapi.tech/api/planets/1,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""arid"", ""surface_water"": ""1"", ""name"": ""Tatooine"", ""diameter"": ""10465"", ""rotation_period"": ""23"", ""terrain"": ""desert"", ""gravity"": ""1 standard"", ""orbital_period"": ""304"", ""population"": ""200000"", ""url"": ""https://www.swapi.tech/api/planets/1""}"
2025-03-15 22:48:11.471530,2025-03-15 22:47:44.286978,Glee Anselm,44,https://www.swapi.tech/api/planets/44,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""tropical, temperate"", ""surface_water"": ""80"", ""name"": ""Glee Anselm"", ""diameter"": ""15600"", ""rotation_period"": ""33"", ""terrain"": ""lakes, islands, swamps, seas"", ""gravity"": ""1"", ""orbital_period"": ""206"", ""population"": ""500000000"", ""url"": ""https://www.swapi.tech/api/planets/44""}"
2025-03-15 22:48:11.471530,2025-03-15 22:47:44.286978,Umbara,60,https://www.swapi.tech/api/planets/60,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""unknown"", ""surface_water"": ""unknown"", ""name"": ""Umbara"", ""diameter"": ""unknown"", ""rotation_period"": ""unknown"", ""terrain"": ""unknown"", ""gravity"": ""unknown"", ""orbital_period"": ""unknown"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/60""}"
2025-03-15 22:48:11.471530,2025-03-15 22:47:44.286978,Vulpter,39,https://www.swapi.tech/api/planets/39,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate, artic"", ""surface_water"": ""unknown"", ""name"": ""Vulpter"", ""diameter"": ""14900"", ""rotation_period"": ""22"", ""terrain"": ""urban, barren"", ""gravity"": ""1"", ""orbital_period"": ""391"", ""population"": ""421000000"", ""url"": ""https://www.swapi.tech/api/planets/39""}"
2025-03-15 22:48:11.471530,2025-03-15 22:47:44.286978,Mon Cala,31,https://www.swapi.tech/api/planets/31,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""100"", ""name"": ""Mon Cala"", ""diameter"": ""11030"", ""rotation_period"": ""21"", ""terrain"": ""oceans, reefs, islands"", ""gravity"": ""1"", ""orbital_period"": ""398"", ""population"": ""27000000000"", ""url"": ""https://www.swapi.tech/api/planets/31""}"
2025-03-15 22:48:11.471530,2025-03-15 22:47:44.286978,Chandrila,32,https://www.swapi.tech/api/planets/32,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""40"", ""name"": ""Chandrila"", ""diameter"": ""13500"", ""rotation_period"": ""20"", ""terra

No. Rows: 78


# 6 Clean Up

In [27]:
import shutil
shutil.rmtree(f"D:/Data/{CATALOG}")